# Clip Dino SAM WSSS pipeline


The intuition behind this pipeline is that
- CLIP works best when the object is centered in the image. CLIP-ES, the variant we use, partially addresses this limitation, but the resulting maps are still fuzzy and not very well aligned to object boundaries.
- DINOv3 is good at finding objects within an image, and its features give cleaner edges than CLIP-ES.
- By producing Class Activation Maps (CAMs) from both CLIP-ES and DINOv3 and combining them, we hope to see an improvement in mIoU.
- To combine the CLIP-ES output with the DINOv3 output, we use patch affinities, which prior work has shown to be effective. A patch is a small region of the image, and patch affinity is the pairwise cosine similarity between patch features. The idea is that if both DINOv3 and CLIP-ES agree on a region, the resulting CAM should be more accurate.
- Finally, a SAM2 block is added at the end of the pipeline to refine the edges of the DINOv3 + CLIP-ES output.

In [ ]:
import sys
import os
import json
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
from lxml import etree
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from dataset import VOC_CLASSES, VOC_PALETTE, WSSSDataset, wsss_collate_fn, make_transform, make_voc_datasets

CLIP_ES_DIR = os.path.abspath("clip_es")
if CLIP_ES_DIR not in sys.path:
    sys.path.insert(0, CLIP_ES_DIR)

import clip
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import scale_cam_image
from clip_text import BACKGROUND_CATEGORY, class_names, new_class_names
from utils import parse_xml_to_dict, scoremap2bbox
import generate_cams_voc12 as ces
from generate_cams_voc12 import zeroshot_classifier, reshape_transform, ClipOutputTarget, _transform_resize

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

from cpm import cpm_from_cams
from eval_cam import _fast_hist

## Step 1: Using CLIP-ES to create labeled CAMs.

The first stage of our pipeline is [CLIP-ES (CVPR 2023)](https://github.com/linyq2117/CLIP-ES).

CLIP-ES uses a frozen CLIP ViT-B/16 with Grad-CAM hooked into the last attention block. This turns image-level labels into CAMs, which are then refined by class-aware attention affinity (CAA) using the model's self attention.

The main advantage of CLIP here is that it doesn't require the object to be centred. Given a text prompt for a class label, it can pick out the regions of the image that the prompt activates.


As for the CLIP-ES implementation itself, we reuse many pieces of code from the CLIP-ES repository. This will be commented in the code below:

In [ ]:
# clip-es has its own classes for better segmentation against PASCAL_VOC.
print("Foreground classes :", new_class_names)
print("Background classes :", BACKGROUND_CATEGORY)

In [ ]:
if torch.cuda.is_available():
    clip_device = "cuda"
elif torch.backends.mps.is_available():
    clip_device = "cpu"
else:
    clip_device = "cpu"

CLIP_MODEL_NAME = "ViT-B/16"
print(f"Loading CLIP-ES on {CLIP_MODEL_NAME}")
clip_model, _ = clip.load(CLIP_MODEL_NAME, device=clip_device)
clip_model.eval()
print("CLIP loaded")
ces.device = clip_device

PROMPT_TEMPLATES = ["a clean origami {}."] # clip-es empirically tested prompt choice
fg_text_features = zeroshot_classifier(new_class_names,    PROMPT_TEMPLATES, clip_model)
bg_text_features = zeroshot_classifier(BACKGROUND_CATEGORY, PROMPT_TEMPLATES, clip_model)

# clip-es itself implements a GradCAM
target_layers = [clip_model.visual.transformer.resblocks[-1].ln_1]
cam = GradCAM(
    model=clip_model,
    target_layers=target_layers,
    reshape_transform=reshape_transform,
)

In [ ]:
# These helpers are re-implemented from the original CLIP-ES repository.

VOC_ROOT = "data/VOCdevkit/VOC2012"
PATCH = 16


def run_clip_es_on_image(image_id):
    img_path = os.path.join(VOC_ROOT, "JPEGImages", f"{image_id}.jpg")
    xml_path = os.path.join(VOC_ROOT, "Annotations", f"{image_id}.xml")

    with open(xml_path) as f:
        data = parse_xml_to_dict(etree.fromstring(f.read()))["annotation"]
    ori_W, ori_H = int(data["size"]["width"]), int(data["size"]["height"])
    present_names = []
    for obj in data["object"]:
        if obj["name"] not in present_names:
            present_names.append(obj["name"])

    label_id_list = [class_names.index(n) for n in present_names]
    if not label_id_list:
        raise RuntimeError(f"{image_id} has no labeled objects")

    h = int(np.ceil(ori_H / PATCH) * PATCH)
    w = int(np.ceil(ori_W / PATCH) * PATCH)
    image = _transform_resize(h, w)(Image.open(img_path)).unsqueeze(0).to(clip_device)
    image_features, attn_weight_list = clip_model.encode_image(image, h, w)

    fg_temp = fg_text_features[label_id_list]
    text_temp = torch.cat([fg_temp, bg_text_features], dim=0).to(clip_device)
    cam_input = [image_features, text_temp, h, w]

    coarse_cams = {}
    refined_cams = {}
    for col_idx, class_idx in enumerate(label_id_list):
        targets = [ClipOutputTarget(col_idx)]
        grayscale_cam, _logits, attn_weight_last = cam(
            input_tensor=cam_input, targets=targets, target_size=None
        )
        grayscale_cam = grayscale_cam[0]
        coarse_cams[class_idx] = cv2.resize(grayscale_cam, (ori_W, ori_H))

        all_attn = attn_weight_list + [attn_weight_last]
        attn = torch.stack([a[:, 1:, 1:] for a in all_attn], dim=0)[-8:]
        attn = attn.mean(dim=0)[0].detach().float().cpu()

        boxes, n_box = scoremap2bbox(grayscale_cam, threshold=0.4, multi_contour_eval=True)
        aff_mask = torch.zeros(grayscale_cam.shape)
        for i in range(n_box):
            x0, y0, x1, y1 = boxes[i]
            aff_mask[y0:y1, x0:x1] = 1
        aff_mask = aff_mask.view(1, -1)

        trans = attn / attn.sum(dim=0, keepdim=True)
        trans = trans / trans.sum(dim=1, keepdim=True)
        for _ in range(2):
            trans = trans / trans.sum(dim=0, keepdim=True)
            trans = trans / trans.sum(dim=1, keepdim=True)
        trans = (trans + trans.T) / 2
        trans = trans @ trans
        trans = trans * aff_mask

        refined = trans @ torch.tensor(grayscale_cam).view(-1, 1)
        refined = refined.view(h // PATCH, w // PATCH).numpy().astype(np.float32)
        refined_cams[class_idx] = scale_cam_image([refined], (ori_W, ori_H))[0]

    n_fg = len(fg_temp)
    bg_cam_per_cat = {}
    for bg_local_idx in range(len(bg_text_features)):
        targets = [ClipOutputTarget(n_fg + bg_local_idx)]
        grayscale_cam, _, _ = cam(
            input_tensor=cam_input, targets=targets, target_size=None
        )
        bg_cam_per_cat[BACKGROUND_CATEGORY[bg_local_idx]] = cv2.resize(
            grayscale_cam[0], (ori_W, ori_H)
        )
    bg_cam = np.maximum.reduce(list(bg_cam_per_cat.values()))
    if bg_cam.max() > 0:
        bg_cam = bg_cam / bg_cam.max()

    return {
        "image_path": img_path,
        "image_size": (ori_W, ori_H),
        "labels": label_id_list,
        "coarse": coarse_cams,
        "refined": refined_cams,
        "bg_cam": bg_cam.astype(np.float32),
        "bg_per_cat": bg_cam_per_cat,
    }


def show_cams(result, alpha=0.55):
    img = np.array(Image.open(result["image_path"]).convert("RGB"))
    n = len(result["labels"])
    fig, axes = plt.subplots(2, n + 1, figsize=(4 * (n + 1), 8))
    if n == 1:
        axes = axes.reshape(2, 2)
    axes[0, 0].imshow(img); axes[0, 0].set_title("input"); axes[0, 0].axis("off")
    axes[1, 0].imshow(img); axes[1, 0].set_title("input"); axes[1, 0].axis("off")
    for i, c in enumerate(result["labels"]):
        for row, key, label in [(0, "coarse", "Grad-CAM"), (1, "refined", "CAA-refined")]:
            heat = cv2.applyColorMap((result[key][c] * 255).astype(np.uint8), cv2.COLORMAP_JET)
            heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
            overlay = (alpha * heat + (1 - alpha) * img).clip(0, 255).astype(np.uint8)
            axes[row, i + 1].imshow(overlay)
            axes[row, i + 1].set_title(f"{label}: {new_class_names[c].split(' ')[0]}")
            axes[row, i + 1].axis("off")
    plt.tight_layout(); plt.show()


sample_id = "2007_000032"
print(f"Running CLIP-ES on {sample_id}...")
result = run_clip_es_on_image(sample_id)
print("Predicted classes (label ids):", result["labels"])
show_cams(result)

# Step 2: Using DINOv3 to generate better CAMs

This step uses DINOv3 features to refine the CLIP-ES heatmap from Step 1 via patch affinities.

Patch affinities assign a similarity score to every pair of patches in the image. The idea is to spread the existing CLIP-ES CAM along directions where DINOv3 features agree, and dampen it where they don't, which helps the CAM stick to object boundaries.

This affinity-based refinement was popularized by a [2018 CVPR paper](https://arxiv.org/abs/1803.10464) ([code](https://github.com/jiwoon-ahn/psa)), and our implementation is based on theirs. The same idea was later adopted in the [2022 CVPR MCTformer paper](https://arxiv.org/abs/2203.02891) ([code](https://github.com/xulianuwa/MCTformer)). The novel part here is using DINOv3 patch tokens, rather than a trained affinity network, as the source of the affinities.


In [ ]:
# loading DINO
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading DINOv3 (ViT-Small)...")
dino_repo_dir = "./dinov3"
dino_model = torch.hub.load(
    dino_repo_dir,
    "dinov3_vits16",
    source="local",
    weights="weights/dinov3_vits16_pretrain_lvd1689m-08c60483.pth",
).to(device)
dino_model.eval()

In [ ]:
PATCH_DINO = 16

@torch.no_grad()
def dino_patch_affinity(image_path, h, w, dino_model, device, threshold=0.2):
    """
    Compute patch affinity pairwise matrix.

    We simply take the cosine similarity between the patch tokens of the dino model.
    """
    img = Image.open(image_path)
    x = make_transform((h, w))(img).unsqueeze(0).to(device)

    out = dino_model.forward_features(x)
    tokens = out["x_norm_patchtokens"][0].float()
    tokens = F.normalize(tokens, dim=-1)

    affinity = (tokens @ tokens.T - threshold).clamp(min=0)
    affinity = affinity / (affinity.sum(dim=1, keepdim=True) + 1e-8)
    return affinity


@torch.no_grad()
def dino_refine(result, dino_model, device, cam_threshold=0.5, n_iter=2):
    """
    Mirrors CLIP-ES's CAA step, but uses DINO cosine-affinity
    """
    ori_W, ori_H = result["image_size"]
    h = int(np.ceil(ori_H / PATCH_DINO) * PATCH_DINO)
    w = int(np.ceil(ori_W / PATCH_DINO) * PATCH_DINO)
    gh, gw = h // PATCH_DINO, w // PATCH_DINO
    N = gh * gw

    # Compute the DINO patch affinity once per image.
    affinity = dino_patch_affinity(result["image_path"], h, w, dino_model, device).float()

    refined_cams = {}
    for c, cam_full in result["refined"].items():
        # For each class, take the CAA-refined CAM, downsample it onto the
        #  DINO patch grid, and build an aff_mask = (CAM_norm > cam_threshold).
        cam_grid = cv2.resize(cam_full.astype(np.float32), (gw, gh))
        cam_t = torch.from_numpy(cam_grid).to(device).view(N, 1)

        cam_norm = (cam_grid - cam_grid.min()) / (cam_grid.max() - cam_grid.min() + 1e-8)
        aff_mask_bool = cam_norm > cam_threshold
        if not aff_mask_bool.any():
            aff_mask_bool = cam_norm >= cam_norm.max()
        aff_mask = torch.from_numpy(aff_mask_bool.astype(np.float32)).to(device).view(1, N)

        trans = affinity * aff_mask
        trans = trans / (trans.sum(dim=0, keepdim=True) + 1e-8)
        trans = trans / (trans.sum(dim=1, keepdim=True) + 1e-8)
        for _ in range(n_iter):
            trans = trans / (trans.sum(dim=0, keepdim=True) + 1e-8)
            trans = trans / (trans.sum(dim=1, keepdim=True) + 1e-8)

        
        # random walk 
        trans = (trans + trans.T) / 2
        trans = trans @ trans

        # combine with cams
        refined = (trans @ cam_t).view(gh, gw).cpu().numpy().astype(np.float32)
        refined_cams[c] = scale_cam_image([refined], (ori_W, ori_H))[0]

    return {**result, "dino_refined": refined_cams}


def show_cams_with_dino(result, alpha=0.55):
    """Side-by-side: input | Grad-CAM | CAA-refined | DINO-refined for each class."""
    img = np.array(Image.open(result["image_path"]).convert("RGB"))
    rows = [
        ("coarse",       "Grad-CAM"),
        ("refined",      "CAA-refined"),
        ("dino_refined", "DINO-refined"),
    ]
    n = len(result["labels"])
    fig, axes = plt.subplots(len(rows), n + 1, figsize=(4 * (n + 1), 4 * len(rows)))
    for r in range(len(rows)):
        axes[r, 0].imshow(img); axes[r, 0].set_title("input"); axes[r, 0].axis("off")
    for i, c in enumerate(result["labels"]):
        for r, (key, label) in enumerate(rows):
            heat = cv2.applyColorMap((result[key][c] * 255).astype(np.uint8), cv2.COLORMAP_JET)
            heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
            overlay = (alpha * heat + (1 - alpha) * img).clip(0, 255).astype(np.uint8)
            axes[r, i + 1].imshow(overlay)
            axes[r, i + 1].set_title(f"{label}: {new_class_names[c].split(' ')[0]}")
            axes[r, i + 1].axis("off")
    plt.tight_layout(); plt.show()

result_with_dino = dino_refine(result, dino_model, device)
show_cams_with_dino(result_with_dino)

From this spot check, we can see that the CAMs are being refined. The edges are still a bit rough, but this was as far as we got. Better ways of integrating CLIP-ES with DINO would be a good direction for future work.

## Step 3: Applying SAM2 to make masks

To use SAM2, we have to feed it some point prompts (seeds) from the image. Given good seeds, SAM2 should produce a clean segmentation of the object.

To generate these seeds from our CAMs, we follow the method from the [IEEE 2024 paper S2C](https://ieeexplore.ieee.org/document/10658447) ([code](https://github.com/sangrockEG/S2C)).

Specifically, we use their "CPM" block, which "extracts prompts from the CAM of each class and uses them to generate class-specific segmentation masks through SAM". It picks point prompts from the CAM by taking the strongest peak plus a few additional well-separated seeds.


In [ ]:
# Loading SAM2
SAM_CONFIG     = "configs/sam2.1/sam2.1_hiera_l.yaml"
SAM_CHECKPOINT = "weights/sam2.1_hiera_large.pt"

sam_model     = build_sam2(SAM_CONFIG, SAM_CHECKPOINT, device=str(device))
sam_predictor = SAM2ImagePredictor(sam_model)
print("SAM2 ready")

In [ ]:
# The CPM method comes from https://github.com/sangrockEG/S2C/blob/main/models/model_s2c.py
# See cpm.py for more details.

NUM_FG_CLASSES = len(VOC_CLASSES)


def show_cpm(result, cpm_result, alpha=0.55, num_fg_classes=NUM_FG_CLASSES):
    """Plot input image, per-class SAM mask with its prompt points, and the
    aggregated pseudo-label from CPM.
    """
    img = np.array(Image.open(result["image_path"]).convert("RGB"))
    n = len(result["labels"])

    fig, axes = plt.subplots(2, n + 1, figsize=(4 * (n + 1), 8))
    axes[0, 0].imshow(img); axes[0, 0].set_title("input"); axes[0, 0].axis("off")

    pgt = cpm_result.pgt
    bg_label = num_fg_classes
    pgt_rgb = np.zeros((*pgt.shape, 3), dtype=np.uint8)
    palette = (np.array(plt.get_cmap("tab20").colors) * 255).astype(np.uint8)
    for c in result["labels"]:
        pgt_rgb[pgt == c] = palette[c % len(palette)]
    pgt_overlay = (alpha * pgt_rgb + (1 - alpha) * img).clip(0, 255).astype(np.uint8)
    axes[1, 0].imshow(pgt_overlay)
    axes[1, 0].set_title(f"CPM pseudo-label (bg={bg_label})")
    axes[1, 0].axis("off")

    for i, c in enumerate(result["labels"]):
        mask = cpm_result.masks[c]
        pts = cpm_result.points[c]
        conf = cpm_result.confs[c]

        mask_rgb = np.zeros_like(img)
        mask_rgb[mask] = (255, 64, 64)
        overlay = (0.55 * mask_rgb + 0.45 * img).clip(0, 255).astype(np.uint8)

        axes[0, i + 1].imshow(img)
        axes[0, i + 1].set_title(new_class_names[c].split(" ")[0])
        axes[0, i + 1].axis("off")

        axes[1, i + 1].imshow(overlay)
        axes[1, i + 1].scatter(pts[:, 0], pts[:, 1], s=80, marker="*", c="lime",
                               edgecolors="black", linewidths=0.8)
        axes[1, i + 1].set_title(f"SAM (conf={conf:.2f}, {len(pts)} pts)")
        axes[1, i + 1].axis("off")

    plt.tight_layout()
    plt.show()


def sam_from_cams(
    result,
    predictor,
    source="dino_refined",
    *,
    th_multi=0.5,
    min_distance=20,
    idx_max_sam=2,
    num_classes=NUM_FG_CLASSES,
):
    cpm_out = cpm_from_cams(
        image=result["image_path"],
        cams=result[source],
        predictor=predictor,
        num_classes=num_classes,
        th_multi=th_multi,
        min_distance=min_distance,
        idx_max_sam=idx_max_sam,
    )

    sam_prompts = {
        c: {"points": cpm_out.points[c], "score": cpm_out.confs[c]}
        for c in cpm_out.masks
    }

    return {
        **result,
        "sam_source": source,
        "sam_masks": cpm_out.masks,
        "sam_prompts": sam_prompts,
        "cpm": cpm_out,
    }


print(f"Running on {sample_id}...")
cpm_out = cpm_from_cams(
    image=result_with_dino["image_path"],
    cams=result_with_dino["dino_refined"],
    predictor=sam_predictor,
    num_classes=NUM_FG_CLASSES,
    th_multi=0.5,
    min_distance=20,
    idx_max_sam=2,
)

show_cpm(result_with_dino, cpm_out)

## Step 4: mIoU evaluation on VOC 2012 val

We partly reuse the CLIP-ES repository evaluation code for VOC 2012. We adapt that evaluation loop to our pipeline and also save per-image masks and comparison plots

In [ ]:
VAL_LIST_PATH = os.path.join(CLIP_ES_DIR, "voc12", "val.txt")
SEG_GT_DIR    = os.path.join(VOC_ROOT, "SegmentationClass")
N_CLASS       = 21  # 20 VOC + bg


def save_pred_png(pred, path):
    """Save a (H, W) uint8 prediction as a VOC-palette PNG, matching the GT format.
    """
    im = Image.fromarray(pred.astype(np.uint8), mode="P")
    im.putpalette(VOC_PALETTE.tobytes())
    im.save(path)


def _class_legend(ax, mask):
    """Add a small legend listing the foreground classes present in `mask`."""
    present = [int(v) for v in np.unique(mask) if 1 <= int(v) <= len(VOC_CLASSES)]
    if not present:
        return
    handles = [Patch(color=VOC_PALETTE[c] / 255.0, label=VOC_CLASSES[c - 1])
               for c in present]
    ax.legend(handles=handles, loc="lower right", fontsize=8,
              framealpha=0.85, handlelength=1.0, handleheight=1.0)


def save_comparison_plot(image_path, gt, preds_by_source, save_path,
                         ious_by_source=None, alpha=0.55):
    """Save a side-by-side figure of the input image, ground truth, and one
    overlay per CAM source
    """
    img = np.array(Image.open(image_path).convert("RGB"))
    ious_by_source = ious_by_source or {}

    def overlay(mask):
        rgb = VOC_PALETTE[np.where(mask == 255, 0, mask)]
        return (alpha * rgb + (1 - alpha) * img).clip(0, 255).astype(np.uint8)

    n_cols = 2 + len(preds_by_source)
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

    axes[0].imshow(img); axes[0].set_title("input"); axes[0].axis("off")

    axes[1].imshow(overlay(gt))
    axes[1].set_title("ground truth"); axes[1].axis("off")
    _class_legend(axes[1], gt)

    for i, (source, pred) in enumerate(preds_by_source.items()):
        ax = axes[2 + i]
        ax.imshow(overlay(pred))
        title = f"{source}+SAM"
        if source in ious_by_source:
            title += f"\nmIoU={ious_by_source[source]:.3f}"
        ax.set_title(title); ax.axis("off")
        _class_legend(ax, pred)

    plt.tight_layout()
    fig.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close(fig)


def load_val_ids(path=VAL_LIST_PATH, subset=None):
    ids = [ln.strip() for ln in open(path) if ln.strip()]
    return ids[:subset] if subset else ids


def predict_per_image(result, source, cam_eval_thres):
    """
    Per-image (H, W) uint8 prediction in 0..20 (0 = bg).
    """
    keys = np.array(result["labels"], dtype=np.int64)
    cams = np.stack([result[source][c] for c in keys]).astype(np.float32)

    masks = np.stack([result["sam_masks"][c] for c in keys]).astype(np.float32)
    cams = cams * masks

    if cam_eval_thres < 1:
        cams = np.pad(cams, ((1, 0), (0, 0), (0, 0)),
                      mode="constant", constant_values=cam_eval_thres)
    else:
        bg = np.power(1.0 - cams.max(axis=0, keepdims=True), cam_eval_thres)
        cams = np.concatenate([bg, cams], axis=0)

    keys_full = np.pad(keys + 1, (1, 0), mode="constant")  # 0 = bg
    return keys_full[cams.argmax(axis=0)].astype(np.uint8)


def metrics_from_hist(hist):
    """Same metrics as eval_cam.scores, but operates on an already-accumulated
    confusion matrix so we can update it one image at a time instead of holding
    every prediction in memory.
    """
    with np.errstate(divide="ignore", invalid="ignore"):
        acc      = np.diag(hist).sum() / hist.sum()
        acc_cls  = np.nanmean(np.diag(hist) / hist.sum(axis=1))
        iu       = np.diag(hist) / (hist.sum(axis=1) + hist.sum(axis=0) - np.diag(hist))
        valid    = hist.sum(axis=1) > 0
        mean_iu  = np.nanmean(iu[valid])
        freq     = hist.sum(axis=1) / hist.sum()
        fwavacc  = (freq[freq > 0] * iu[freq > 0]).sum()
    return {
        "Pixel Accuracy":         acc,
        "Mean Accuracy":          acc_cls,
        "Frequency Weighted IoU": fwavacc,
        "Mean IoU":               mean_iu,
        "Class IoU":              dict(zip(range(hist.shape[0]), iu)),
    }


def evaluate(
    val_ids,
    sources=("refined", "dino_refined"),
    cam_eval_thres=2.0,
    sam_seed_source="dino_refined",
    save_dir=None,
):
    """Run the full pipeline (CLIP-ES, then DINO refinement, then SAM gating)
    on every image, and score each CAM source in `sources` against the GT.
    """
    hists = {s: np.zeros((N_CLASS, N_CLASS), dtype=np.int64) for s in sources}
    per_image: dict[str, dict] = {}
    skipped = []

    if save_dir is not None:
        save_dir = Path(save_dir)
        for s in sources:
            (save_dir / "masks" / s).mkdir(parents=True, exist_ok=True)
        (save_dir / "plots").mkdir(parents=True, exist_ok=True)

    for image_id in tqdm(val_ids, desc="Evaluating"):
        try:
            res = run_clip_es_on_image(image_id)
        except Exception as e:
            skipped.append((image_id, repr(e)))
            continue

        res = dino_refine(res, dino_model, device)
        res = sam_from_cams(res, sam_predictor, source=sam_seed_source)

        gt = np.array(Image.open(os.path.join(SEG_GT_DIR, f"{image_id}.png"))).astype(np.uint8)
        gt_flat = gt.flatten()

        preds_by_source: dict[str, np.ndarray] = {}
        ious_by_source: dict[str, float] = {}
        for source in sources:
            pred = predict_per_image(res, source=source, cam_eval_thres=cam_eval_thres)
            hist_img = _fast_hist(gt_flat, pred.flatten(), N_CLASS)
            hists[source] += hist_img
            preds_by_source[source] = pred
            ious_by_source[source]  = float(metrics_from_hist(hist_img)["Mean IoU"])
            if save_dir is not None:
                save_pred_png(pred, save_dir / "masks" / source / f"{image_id}.png")

        plot_rel = f"plots/{image_id}.png"
        per_image[image_id] = {
            "plot_path": plot_rel,
            "miou": ious_by_source,
        }

        if save_dir is not None:
            save_comparison_plot(
                res["image_path"], gt, preds_by_source,
                save_dir / plot_rel,
                ious_by_source=ious_by_source,
            )

    if skipped:
        print(f"\nSkipped {len(skipped)} image(s); first few: {skipped[:3]}")

    if save_dir is not None:
        (save_dir / "per_image.json").write_text(
            json.dumps(per_image, indent=2, sort_keys=True)
        )
        print(f"Saved masks + plots + per_image.json to {save_dir}/")

    results = {s: metrics_from_hist(h) for s, h in hists.items()}
    return results, hists


def print_results_table(results, cam_eval_thres):
    """Print a per-class IoU table with one column per CAM source (each gated by SAM)."""
    sources = list(results.keys())
    headers = [f"{s}+SAM" for s in sources]
    name_w = max(len(n) for n in (["background"] + VOC_CLASSES))

    print(f"\n=== mIoU report (cam_eval_thres={cam_eval_thres}) ===")
    print("class".ljust(name_w) + " | " + " | ".join(h.center(14) for h in headers))
    print("-" * (name_w + 3 + 17 * len(headers)))
    for c, name in enumerate(["background"] + VOC_CLASSES):
        row = [f"{results[s]['Class IoU'][c]:.4f}".center(14) for s in sources]
        print(name.ljust(name_w) + " | " + " | ".join(row))
    print("-" * (name_w + 3 + 17 * len(headers)))

    for label in ["Mean IoU", "Pixel Accuracy", "Mean Accuracy", "Frequency Weighted IoU"]:
        row = [f"{results[s][label]:.4f}".center(14) for s in sources]
        print(label.ljust(name_w) + " | " + " | ".join(row))


print(f"Found {len(load_val_ids())} val image IDs at {VAL_LIST_PATH}")
print(f"GT segmentation masks at {SEG_GT_DIR}")

In [ ]:
# test progess
SUBSET           = 10
CAM_EVAL_THRES   = 2.0
SAVE_DIR         = "results"

SOURCES = [
    "refined",
    "dino_refined",
]

val_ids = load_val_ids()
# val_ids = load_val_ids(subset=SUBSET)

results, hists = evaluate(
    val_ids,
    sources=SOURCES,
    cam_eval_thres=CAM_EVAL_THRES,
    save_dir=SAVE_DIR,
)
print_results_table(results, CAM_EVAL_THRES)


# Evaluation

The evaluation is rather disappointing. While we expected the DINO refined pipeline to improve upon the CLIP-ES results, in most cases it ended up working against it, with the mIoU of the DINO-free pipeline largely outperforming the DINO pipeline.

That being said, the results are nonetheless better than the base case pipeline. This is expected due to the stronger capabilities of CLIP-ES and SAM.

Further steps with this pipeline would be to refine and review approaches of combining DINO with CLIP-ES.